<a href="https://colab.research.google.com/github/RMoulla/IAA_Lyon/blob/main/RAG_Gradio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Application RAG avec Gradio**

In [ ]:
pip install gradio openai pypdf faiss-cpu sentence-transformers

## Setup OpenAI API Key

To use the OpenAI API, you need an API key. Please add your OpenAI API key to the Colab secrets manager under the name `OPENAI_API_KEY`.

In [ ]:
# Import necessary libraries
import gradio as gr
import openai
import os
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

# Securely load the OpenAI API key
from google.colab import userdata

openai.api_key = userdata.get('OPENAI_API_KEY')

## 1. Load and Process PDFs

This section will handle loading PDF documents from a specified directory, splitting them into manageable chunks, and preparing them for embedding.

In [ ]:
import os
from pypdf import PdfReader

def load_pdfs_from_directory(base_directory):
    docs = []
    for root, _, files in os.walk(base_directory):
        for filename in files:
            if filename.endswith('.pdf'):
                filepath = os.path.join(root, filename)
                try:
                    reader = PdfReader(filepath)
                    for page in reader.pages:
                        docs.append(page.extract_text())
                except Exception as e:
                    print(f"Error reading {filename} at {filepath}: {e}. Skipping.")
    return docs

def split_text_into_chunks(text, chunk_size=500, overlap=50):
    chunks = []
    words = text.split()
    i = 0
    while i < len(words):
        chunk = ' '.join(words[i:i + chunk_size])
        chunks.append(chunk)
        i += chunk_size - overlap
    return chunks

# Ensure the 'pdfs' directory exists (it's still good practice to have it, even if not exclusively used)
if not os.path.exists('pdfs'):
    os.makedirs('pdfs')

# Remove any potentially invalid 'sample.pdf' created by previous runs in the 'pdfs' subdirectory
sample_pdf_path = os.path.join('pdfs', 'sample.pdf')
if os.path.exists(sample_pdf_path):
    os.remove(sample_pdf_path)
    print("Removed old, potentially invalid 'sample.pdf' from 'pdfs/' directory.")

# Load and process documents from the /content/ directory (the current working directory)
pdf_directory = '/content/'
raw_documents = load_pdfs_from_directory(pdf_directory)
all_chunks = []
for doc in raw_documents:
    all_chunks.extend(split_text_into_chunks(doc))

print(f"Loaded {len(raw_documents)} raw PDF pages and split into {len(all_chunks)} chunks.")
if len(raw_documents) == 0:
    print("No valid PDF files found in the /content/ directory or its subdirectories.")


## 2. Create Embeddings and Vector Store

Here, we'll convert the text chunks into numerical embeddings using a `SentenceTransformer` model and then store them in a FAISS index for efficient similarity search.

In [ ]:
# Initialize sentence transformer model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Generate embeddings for all chunks
chunk_embeddings = model.encode(all_chunks)

# --- DEBUG: Print chunk_embeddings to inspect its content ---
print(f"Shape of chunk_embeddings: {chunk_embeddings.shape}")
print(f"First 5 embeddings (if any):\n{chunk_embeddings[:5]}")
# ---------------------------------------------------------

# Create a FAISS index
dimension = chunk_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(chunk_embeddings))

print(f"FAISS index created with {index.ntotal} embeddings.")

## 3. RAG Function

This function will take a user query, retrieve relevant document chunks from the FAISS index, and then use the OpenAI API to generate a coherent answer based on the retrieved context.

In [ ]:
def rag_query(query, k=3):
    # 1. Embed the query
    query_embedding = model.encode([query])

    # 2. Search FAISS index for relevant chunks
    distances, indices = index.search(np.array(query_embedding), k)

    # 3. Retrieve relevant document chunks
    retrieved_chunks = [all_chunks[i] for i in indices[0]]
    context = "\n\n".join(retrieved_chunks)

    # 4. Use OpenAI API to generate response based on context
    try:
        response = openai.chat.completions.create(
            model="gpt-4o",
            messages=[
                {"role": "system", "content": "You are a helpful assistant that answers questions based on the provided context."},
                {"role": "user", "content": f"Context: {context}\n\nQuestion: {query}\n\nAnswer:"}
            ]
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"An error occurred with the OpenAI API: {e}"


## 4. Gradio Interface

Finally, we'll create a user-friendly web interface using Gradio to interact with our RAG system.

In [ ]:
def gradio_interface(query):
    return rag_query(query)

iface = gr.Interface(fn=gradio_interface, inputs="text", outputs="text", title="RAG Chatbot with PDFs", description="Ask questions about the content of the uploaded PDFs.")
iface.launch(debug=True)